# Part 2 — Notebook 09: Memory — Trí nhớ AI giữa các phiên

**Vibe Coding Guide cho người KHÔNG biết lập trình**

---

> Claude Code mỗi conversation = phiên MỚI, KHÔNG nhớ phiên trước. Memory files = cách AI "nhớ".

## Memory là gì?

Hãy tưởng tượng bạn điều hành một **nhà hàng** với ca đêm và ca ngày.

**Không có sổ bàn giao:**
- Ca ngày đặt hàng thêm 50kg thịt bò, ca đêm không biết → đặt thêm 50kg nữa
- Ca ngày phát hiện bếp 3 bị hỏng, ca đêm dùng bếp 3 → cháy!
- Ca ngày đổi menu soup, ca đêm vẫn nấu soup cũ

**Có sổ bàn giao:**
- "Đã đặt 50kg thịt bò, giao ngày mai" → ca đêm không đặt lại
- "Bếp 3 hỏng, đang sửa" → ca đêm tránh bếp 3
- "Menu soup mới: khoai tây" → ca đêm nấu đúng

**Memory = sổ bàn giao** cho Claude Code.

## Vấn đề: Claude Code "mất trí nhớ"

Mỗi lần bạn mở Claude Code = 1 phiên (conversation) hoàn toàn mới.

| Phiên | AI biết gì |
|-------|------------|
| Phiên 1 | Bạn dạy AI: "Dùng port 5174, không phải 3000" |
| Phiên 2 | AI quên sạch! Lại dùng port 3000 |
| Phiên 3 | AI quên sạch! Lại dùng port 3000 |

Bạn phải dạy lại mỗi phiên? Không! Memory files giải quyết vấn đề này.

Khi có memory:
| Phiên | AI biết gì |
|-------|------------|
| Phiên 1 | Bạn dạy AI, AI ghi vào memory: "Port 5174, không phải 3000" |
| Phiên 2 | AI đọc memory → biết ngay dùng port 5174 |
| Phiên 3 | AI đọc memory → biết ngay dùng port 5174 |

## Cấu trúc Memory

Memory sống ở 2 nơi:

### 1. Project Memory (trong dự án)
```
.claude/
└── memory/
    └── MEMORY.md      <- File chính, AI đọc khi bắt đầu
```

### 2. User Memory (cá nhân, ngoài dự án)
```
~/.claude/
└── projects/
    └── {project-path}/
        └── memory/
            └── MEMORY.md
```

- **Project memory:** Ai cũng thấy (commit vào git), chứa thông tin chung của dự án
- **User memory:** Chỉ bạn thấy (trên máy bạn), chứa thông tin cá nhân

In [ ]:
# Xem cấu trúc MEMORY.md

cat << 'MEMORY'
# Project Name — Claude Memory

**Project:** /path/to/my-project
**Last Updated:** 2026-04-01

---

## PROJECT STATE

| Service | Progress | Status |
|---------|----------|--------|
| Backend | 80%      | In Progress |
| Frontend| 50%      | In Progress |
| Deploy  | 0%       | Not Started |

**Next task:** Frontend login page

---

## CRITICAL PATTERNS

### Port Configuration
- Dev server: port 5174 (NOT 3000)
- API server: port 8080

### Database
- Always set TenantContext before DB operations
- Never call .instanceId() on builder

---

## ENVIRONMENT

- Java 21 at /home/user/.local/java/jdk-21
- pnpm at /home/user/.npm-global/bin/pnpm
MEMORY

echo ""
echo "--- MEMORY.md = muc luc + thong tin thiet yeu ---"

## MEMORY.md chứa gì?

| Section | Mục đích | Ví dụ |
|---------|---------|-------|
| **Project State** | Tình trạng hiện tại | Backend 80%, Frontend 50% |
| **Critical Patterns** | Bẫy cần tránh | Không gọi .instanceId() trên builder |
| **Environment** | Cài đặt môi trường | Java 21 ở đường dẫn X |
| **Git Workflow** | Quy trình git đang dùng | Branch convention, hooks |

MEMORY.md có giới hạn (~200 dòng) vì AI đọc nó MỖI phiên. Nếu quá dài, AI tốn "bộ nhớ làm việc" (tokens) cho memory thay vì cho công việc.

## 4 Seed Memories — Bài học từ dự án thực

Kit đi kèm 4 bài học (seed memories) rút ra từ kinh nghiệm thực tế. Đây là những sai lầm đã xảy ra và cách tránh:

### 1. Business Design First
**Bài học:** Thiết kế tài liệu nghiệp vụ TRƯỚC khi viết code.

Chuyện gì đã xảy ra?
- AI nhảy thẳng vào code mà không thiết kế trước
- Code xong mới phát hiện logic sai
- Phải xóa code, viết lại từ đầu

**Bây giờ:** AI luôn tạo business docs (rules, use-cases, API contract) TRƯỚC khi code.

### 2. CI Before Scoring
**Bài học:** Không chấm điểm chất lượng khi CI chưa chạy xong.

Chuyện gì đã xảy ra?
- AI chạy quality audit, chấm 90/100
- CI fail vì lỗi compilation
- Score 90 điểm = vô nghĩa, code không chạy được!

**Bây giờ:** AI chờ CI pass rồi mới chạy quality audit.

### 3. Scripts, Not Ad-hoc
**Bài học:** Dùng scripts, không chạy lệnh bừa.

Chuyện gì đã xảy ra?
- AI tự gõ lệnh Docker, quên flag quan trọng
- Container chạy sai cấu hình, debug mất 2 giờ
- Lần sau lại quên flag khác

**Bây giờ:** AI dùng scripts đã được kiểm tra, không bao giờ gõ lệnh trực tiếp.

### 4. Self-Test Before Push
**Bài học:** Test local trước khi push lên GitHub.

Chuyện gì đã xảy ra?
- AI push code → CI fail → sửa → push → CI fail lại → sửa...
- Mỗi vòng CI mất 5-10 phút
- 5 lần fail = 30-50 phút lãng phí

**Bây giờ:** AI chạy `test-local.sh` trước khi push. CI fail giảm 90%.

In [ ]:
# Xem nội dung 1 seed memory

cat << 'SEED'
# Feedback: Scripts, Not Ad-hoc Commands

## What happened
AI ran Docker commands directly, forgot important flags.
Container ran with wrong configuration, took 2 hours to debug.

## Root cause
Ad-hoc commands are error-prone:
- Easy to forget flags
- Different every time
- No validation built in

## Lesson learned
ALWAYS use scripts from scripts/ directory.
NEVER run Docker, test, or deploy commands directly.
Scripts are tested, standardized, and handle edge cases.

## Apply when
- About to run any infrastructure command
- Setting up development environment
- Running tests or builds
SEED

echo ""
echo "--- Moi seed memory = 1 bai hoc tu sai lam thuc te ---"

## Memory hoạt động thế nào?

```
Phiên 1:                          Phiên 2:
┌─────────────────┐               ┌─────────────────┐
│ AI bắt đầu      │               │ AI bắt đầu      │
│                  │               │                  │
│ Đọc CLAUDE.md    │               │ Đọc CLAUDE.md    │
│ Đọc MEMORY.md    │               │ Đọc MEMORY.md    │
│                  │               │ -> Biết port 5174│
│ Làm việc...      │               │ -> Biết Java 21  │
│                  │               │ -> Biết bẫy cần  │
│ Phát hiện port   │               │    tránh         │
│ 5174 mới đúng    │               │                  │
│                  │               │ Làm việc đúng    │
│ Ghi vào MEMORY   │               │ ngay từ đầu!     │
└─────────────────┘               └─────────────────┘
```

Memory là cầu nối giữa các phiên — AI phiên sau kế thừa kiến thức từ phiên trước.

## Cập nhật Memory

Memory được cập nhật bằng 2 cách:

### Cách 1: AI tự cập nhật
Khi AI phát hiện điều quan trọng, nó ghi vào memory:
```
AI: "Tôi phát hiện port 5174 mới đúng. Ghi vào memory."
```

### Cách 2: Bạn bảo AI ghi
Khi bạn muốn AI nhớ điều gì đó:
```
Bạn: "Ghi nhớ: luôn dùng dark mode khi chụp screenshot"
AI: "Đã ghi vào memory."
```

Cả 2 cách đều update file MEMORY.md.

In [ ]:
# Mô phỏng AI cập nhật memory
echo "=== Phiên làm việc ==="
echo ""
echo "Bạn: 'Chạy dev server'"
echo "AI: Đang chạy trên port 3000..."
echo "AI: Lỗi! Port 3000 đã bị chiếm."
echo "AI: Thử port 5174... Thành công!"
echo ""
echo "AI: Phát hiện port 5174 là port đúng."
echo "AI: Ghi vào MEMORY.md:"
echo ""
echo "  ## Environment"
echo "  - Dev server: port 5174 (NOT 3000 - port 3000 bị chiếm bởi service khác)"
echo ""
echo "--- Phiên sau, AI sẽ đọc memory và dùng port 5174 ngay ---"

## Giới hạn và best practices

| Quy tắc | Lý do |
|---------|-------|
| MEMORY.md tối đa ~200 dòng | AI đọc mỗi phiên → quá dài = tốn token |
| Mỗi entry ngắn gọn (1-2 dòng) | Dễ scan, không cần đọc kỹ |
| Chi tiết để trong file riêng | MEMORY.md = mục lục, file khác = nội dung |
| Xóa thông tin không còn đúng | Memory cũ = memory sai |

Ví dụ **tốt** (ngắn gọn):
```
- Dev server: port 5174 (NOT 3000)
- Java 21 at /home/user/.local/java/jdk-21
```

Ví dụ **xấu** (quá chi tiết):
```
- Ngày 15/3, khi chạy dev server, ban đầu tôi thử port 3000
  nhưng không được vì service X đã chiếm port đó. Sau khi thử
  nhiều port khác nhau (3001, 5173, 5174), cuối cùng port 5174
  hoạt động tốt nhất vì... (10 dòng giải thích)
```

## Bạn (non-dev) cần biết gì?

1. **Memory = AI nhớ giữa các phiên** — bạn không cần dạy lại mỗi lần
2. **Bạn có thể bảo AI ghi nhớ** — "Ghi nhớ: luôn dùng tiếng Việt"
3. **AI học từ sai lầm** — sai 1 lần → ghi memory → không sai lần 2
4. **Kit có sẵn 4 bài học** — bạn không cần phải sai trước mới có memory
5. **Memory tích lũy** — dự án càng lâu, AI càng "khôn"

Sau 1-2 tháng làm việc, memory file sẽ chứa đầy bài học quý giá — đây là **tài sản** của dự án.

## Ví von tổng kết

| Đời thực | Memory |
|----------|--------|
| Sổ bàn giao ca | MEMORY.md |
| Bài học trong sổ | Seed memories |
| "Bếp 3 hỏng, tránh" | "Port 3000 bị chiếm, dùng 5174" |
| "Menu mới: khoai tây" | "Java 21, không phải Java 17" |
| Đổi ca mà không mất thông tin | Đổi phiên mà AI vẫn nhớ |

---

**Tóm tắt Notebook 09:**

- Memory = "sổ bàn giao" cho AI giữa các phiên làm việc
- MEMORY.md = file chính, AI đọc mỗi khi bắt đầu
- 4 seed memories: business-first, CI-before-scoring, scripts-not-adhoc, test-before-push
- Bạn có thể bảo AI ghi nhớ bất cứ điều gì
- Memory tích lũy theo thời gian = dự án càng lâu, AI càng giỏi
- Giữ MEMORY.md ngắn gọn (~200 dòng), chi tiết để trong file riêng